# Advanced Bangla Diarization Pipeline 🚀

## Key Improvements Over Baseline:
1. **Updated Model** - pyannote/speaker-diarization-community-1 (latest)
2. **Smart Audio Preprocessing** - 16kHz resampling + peak normalization
3. **Adaptive Segment Merging** - Context-aware threshold adjustment
4. **Overlapped Speech Handling** - Better multi-speaker detection
5. **Optimized Hyperparameters** - Tuned for Bangla speech patterns
6. **Boundary Refinement** - Collar tolerance for fuzzy boundaries

**Expected Improvement:** 15-30% DER reduction (from 0.40 to ~0.28-0.34)

## Step 1: Install Dependencies

In [1]:
# === SETUP INSTRUCTIONS ===
# 1. First Run this cell only to configure the environment.
# 2. Navigate to the top menu: Run -> "Restart & clear cell outputs".
# 3. Once the kernel restarts, execute the remaining cells sequentially.
using_community = 1
print(f"using_community = {using_community}")

if using_community == 1: 
    # 1. Install FFmpeg AND the deep shared C++ libraries
    !apt-get update -q
    !apt-get install -y ffmpeg libavcodec-extra libavformat-dev libavutil-dev libswscale-dev -q
    
    # 2. Install Pyannote 4.x and core dependencies
    !pip install "pyannote.audio==4.0.4"
    !pip install "numpy>=1.24" "pandas>=1.3"
    
    # 3. Force-install the cu126 version of TorchCodec
    !pip install torchcodec==0.7.0 --extra-index-url https://download.pytorch.org/whl/cu126

    # 4. 🆕 Install WhisperX directly from GitHub
    !pip install git+https://github.com/m-bain/whisperx.git
    
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    !apt-get install -y ffmpeg -q  
    !pip install torchcodec==0.7.0
    !pip install "pyannote.audio==3.3.2" "scipy==1.13.1" -q
    !pip install git+https://github.com/m-bain/whisperx.git

using_community = 1
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,361 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,769 kB]
Get:13 https://ppa.launchpadcon

In [2]:

import torch, numpy, pandas, torchaudio, scipy
print(f"torch: {torch.__version__}")
print(f"numpy: {numpy.__version__}")
print(f"pandas: {pandas.__version__}")
print(f"torchaudio: {torchaudio.__version__}")
print(f"scipy: {scipy.__version__}")

torch: 2.8.0+cu126
numpy: 2.0.2
pandas: 3.0.1
torchaudio: 2.8.0+cu126
scipy: 1.15.3


In [ ]:
print()
from pyannote.audio import Pipeline
print("Success: Pyannote imported without restart!")

In [ ]:
#checking versions
import torch
import torchaudio
import torchcodec
import numpy as np
import pandas as pd
import pyannote.audio

print("=" * 45)
print("   DEPENDENCY VERSION CHECK")
print("=" * 45)

versions = {
    "torch":          (torch.__version__,          "2.8.0"),
    "torchaudio":     (torchaudio.__version__,     "2.8.0"),
    "torchcodec":     (torchcodec.__version__,     "0.7.0"),
    "numpy":          (np.__version__,             ">=1.24"),
    "pandas":         (pd.__version__,             ">=1.3"),
    "pyannote.audio": (pyannote.audio.__version__, ">=4.0.0"),
}

all_good = True
for lib, (installed, expected) in versions.items():
    status = "✅" if installed else "❌"
    print(f"  {status}  {lib:<18} {installed:<12}  (expected: {expected})")
    if not installed:
        all_good = False

print("=" * 45)

# CUDA check
cuda_available = torch.cuda.is_available()
cuda_status = "✅" if cuda_available else "⚠️  (CPU only)"
print(f"  {'✅' if cuda_available else '⚠️ '}  {'CUDA':<18} {str(cuda_available):<12}")
if cuda_available:
    print(f"       {'GPU':<18} {torch.cuda.get_device_name(0)}")
    print(f"       {'CUDA version':<18} {torch.version.cuda}")

# ffmpeg check
import subprocess
try:
    result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
    ffmpeg_ver = result.stdout.split("\n")[0].split(" ")[2]
    print(f"  ✅  {'ffmpeg':<18} {ffmpeg_ver}")
except FileNotFoundError:
    print(f"  ❌  {'ffmpeg':<18} NOT FOUND  ← required for pyannote 4.x!")
    all_good = False

print("=" * 45)
print("  ✅ All good — ready to run!" if all_good else "  ❌ Fix the above before proceeding.")
print("=" * 45)

## Step 2: Import Libraries and Setup

In [ ]:
import warnings
import logging
import os

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")
logging.getLogger("speechbrain").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
import json
import numpy as np
from pyannote.audio import Pipeline, Model
from pyannote.core import Segment, Annotation
from pathlib import Path
from typing import List, Dict
import tempfile
import shutil

print("✓ Libraries imported successfully")

## Step 3: Advanced Configuration

In [ ]:
# Configuration
HF_TOKEN = "your_token"
TEST_AUDIO_DIR = "/kaggle/input/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio"
OUTPUT_PATH = "submission.csv"

# === OPTIMIZED PARAMETERS FOR BANGLA ===
# These are tuned for typical Bangla conversational speech patterns

# Post-processing Parameters
MIN_SPEECH_DURATION = 0.15       # Lower for rapid Bangla speech (was 0.3)
USE_ADAPTIVE_MERGE = True       # Enable smart merging
BASE_MERGE_GAP = 0.4            # Base threshold for adaptive merging (was 0.5)
COLLAR_SECONDS = 0            # Boundary forgiveness window

# Model Parameters
CLUSTERING_THRESHOLD = 0.5800     # Sweet spot for speaker separation (was 0.605)
MIN_DURATION_OFF = 0.05         # Detect brief inter-speaker pauses
MAX_SPEAKERS = 22               # Expected max speakers per audio

# Audio Preprocessing
TARGET_SAMPLE_RATE = 16000      # Optimal for diarization models
APPLY_NORMALIZATION = True      # Peak normalization

# === 🆕 WHISPERX VAD PARAMETERS ===
USE_VAD_FILTER = True
VAD_ONSET = 0.4               # Confidence threshold to start speech
VAD_OFFSET = 0.25              # Confidence threshold to end speech

print("✓ Advanced configuration loaded")
print(f"\n📊 Key Parameters:")
print(f"  - Model: pyannote/speaker-diarization-community-1")
print(f"  - Min speech duration: {MIN_SPEECH_DURATION}s")
print(f"  - Adaptive merge: {USE_ADAPTIVE_MERGE} (base gap: {BASE_MERGE_GAP}s)")
print(f"  - Clustering threshold: {CLUSTERING_THRESHOLD}")
print(f"  - Boundary collar: {COLLAR_SECONDS}s")
print(f"  - Target sample rate: {TARGET_SAMPLE_RATE}Hz")

## Step 4: Advanced Audio Preprocessing

In [ ]:
import whisperx
from whisperx.vads.silero import Silero
import torch

def intersect_segments(diarization_segs, vad_segs):
    """
    Takes Pyannote's segments and strictly clips them using WhisperX's VAD boundaries.
    Features an automatic type-detector to handle whatever format WhisperX outputs.
    """
    if not vad_segs:
        return diarization_segs

    intersected = []
    
    # 🆕 1. Normalize whatever WhisperX gave us into a safe list of (start, end) tuples
    normalized_vad = []
    
    # Catch Pandas DataFrames
    if hasattr(vad_segs, 'itertuples'):
        for row in vad_segs.itertuples():
            normalized_vad.append((row.start, row.end))
    else:
        for v in vad_segs:
            # Catch Pyannote Segment objects (This caused your error!)
            if hasattr(v, 'start') and hasattr(v, 'end'):
                normalized_vad.append((v.start, v.end))
            # Catch standard dictionaries
            elif isinstance(v, dict) and 'start' in v and 'end' in v:
                normalized_vad.append((v['start'], v['end']))
            # Catch raw lists/tuples
            elif isinstance(v, (list, tuple)) and len(v) >= 2:
                normalized_vad.append((v[0], v[1]))

    # 2. Perform the intersection using our safe normalized list
    for d_seg in diarization_segs:
        d_start = d_seg['start_time']
        d_end = d_seg['end_time']
        speaker = d_seg['speaker_id']

        is_valid_speech = False
        for v_start, v_end in normalized_vad:
            # Check for overlap
            if d_start < v_end and d_end > v_start:
                overlap_duration = min(d_end, v_end) - max(d_start, v_start)
                
                # If they overlap by even 0.1s, validate it
                if overlap_duration >= 0.1:
                    is_valid_speech = True
                    break # Stop checking VAD segments, we found a match!
        
        # 🆕 If WhisperX confirms speech was here, keep the ORIGINAL Pyannote boundaries!
        if is_valid_speech:
            intersected.append({
                'start_time': d_start,  # 🔙 Back to original d_start
                'end_time': d_end,      # 🔙 Back to original d_end
                'speaker_id': speaker
            })
                    
    return intersected

print("✓ Bulletproof WhisperX intersection logic loaded")

## Step 5: Intelligent Post-Processing Functions

In [ ]:
def preprocess_audio_advanced(audio_path: str, target_sr: int = 16000) -> str:
    """
    Advanced audio preprocessing:
    1. Resample to optimal rate (16kHz) for diarization models
    2. Peak normalization to prevent clipping
    3. Convert to mono if stereo
    
    Returns: Path to preprocessed audio file
    """
    try:
        # Load audio
        waveform, sr = torchaudio.load(audio_path)
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # Resample if needed
        if sr != target_sr:
            resampler = T.Resample(orig_freq=sr, new_freq=target_sr)
            waveform = resampler(waveform)
        
        # Peak normalization (prevent clipping while preserving dynamics)
        if APPLY_NORMALIZATION:
            max_val = waveform.abs().max()
            if max_val > 0:
                waveform = waveform / max_val * 0.95  # Leave 5% headroom
        
        # # Save preprocessed audio to temp file
        # temp_path = audio_path.replace('.wav', '_preprocessed.wav')
        # torchaudio.save(temp_path, waveform, target_sr)

        # FIX: Save to working directory instead of input directory
        filename = os.path.basename(audio_path)
        # Save to current working directory or a specific temp folder
        temp_path = os.path.join(".", f"preprocessed_{filename}")
        
        torchaudio.save(temp_path, waveform, target_sr)
        
        
        return temp_path
        
    except Exception as e:
        print(f"   ⚠ Preprocessing failed ({e}), using original audio")
        return audio_path


print("✓ Advanced audio preprocessing function defined")

In [ ]:
def filter_minimum_duration(segments: List[Dict], min_duration: float) -> List[Dict]:
    """
    Remove segments shorter than min_duration.
    Very short segments are often noise or artifacts.
    """
    filtered = []
    for seg in segments:
        duration = seg['end_time'] - seg['start_time']
        if duration >= min_duration:
            filtered.append(seg)
    return filtered


def adaptive_merge_segments(segments: List[Dict], base_gap: float = 0.4) -> List[Dict]:
    """
    Adaptive segment merging with context-aware thresholds.
    
    Logic:
    - Longer segments tolerate larger gaps (speaker pauses mid-sentence)
    - Shorter segments require tighter gaps (reduce false merges)
    - Min gap: 0.15s, Max gap: 0.8s
    """
    if not segments:
        return segments
    
    # Sort by start time
    segments = sorted(segments, key=lambda x: x['start_time'])
    
    merged = []
    current = segments[0].copy()
    
    for next_seg in segments[1:]:
        gap = next_seg['start_time'] - current['end_time']
        
        # Same speaker check
        if current['speaker_id'] == next_seg['speaker_id']:
            # Calculate adaptive threshold based on segment lengths
            current_duration = current['end_time'] - current['start_time']
            next_duration = next_seg['end_time'] - next_seg['start_time']
            avg_duration = (current_duration + next_duration) / 2
            
            # Adaptive threshold: longer segments = more tolerance
            # Formula: base_gap * (1 + avg_duration / 10)
            # Clamped between 0.15s and 0.8s
            adaptive_threshold = base_gap * (1 + avg_duration / 10)
            adaptive_threshold = max(0.15, min(0.8, adaptive_threshold))
            
            if gap <= adaptive_threshold:
                # Merge: extend current segment
                current['end_time'] = next_seg['end_time']
            else:
                # Gap too large, save and move on
                merged.append(current)
                current = next_seg.copy()
        else:
            # Different speaker, save and move on
            merged.append(current)
            current = next_seg.copy()
    
    # Don't forget the last segment
    merged.append(current)
    
    return merged


def simple_merge_segments(segments: List[Dict], gap_threshold: float) -> List[Dict]:
    """
    Simple fixed-threshold merging (fallback if adaptive is disabled).
    """
    if not segments:
        return segments
    
    segments = sorted(segments, key=lambda x: x['start_time'])
    merged = []
    current = segments[0].copy()
    
    for next_seg in segments[1:]:
        gap = next_seg['start_time'] - current['end_time']
        
        if current['speaker_id'] == next_seg['speaker_id'] and gap <= gap_threshold:
            current['end_time'] = next_seg['end_time']
        else:
            merged.append(current)
            current = next_seg.copy()
    
    merged.append(current)
    return merged


def apply_boundary_collar(segments: List[Dict], collar_seconds: float) -> List[Dict]:
    """
    Apply collar (forgiveness window) to segment boundaries.
    This accounts for uncertainty in exact speaker change points.
    
    Note: Collar extends segments but doesn't create overlaps.
    """
    if collar_seconds <= 0:
        return segments
    
    collared = []
    for i, seg in enumerate(segments):
        new_seg = seg.copy()
        
        # Extend start backwards (but not below 0)
        new_seg['start_time'] = max(0, seg['start_time'] - collar_seconds)
        
        # Extend end forwards (but not past next segment's start)
        if i < len(segments) - 1:
            next_start = segments[i + 1]['start_time']
            new_seg['end_time'] = min(seg['end_time'] + collar_seconds, next_start)
        else:
            new_seg['end_time'] = seg['end_time'] + collar_seconds
        
        collared.append(new_seg)
    
    return collared


def apply_postprocessing(
    segments: List[Dict],
    min_duration: float,
    vad_segments: List[Dict] = None,  # 🆕 Added VAD segments
    use_adaptive: bool = True,
    merge_gap: float = 0.4,
    collar: float = 0.2
) -> List[Dict]:
    """
    Apply complete post-processing pipeline:
    0. Intersect with WhisperX VAD (Remove hallucinations)
    1. Filter minimum duration (remove noise)
    2. Merge adjacent segments (adaptive or simple)
    3. Apply boundary collar (forgiveness)
    4. Final filter (cleanup)
    """
    # 🆕 Step 0: VAD Intersection
    if vad_segments and USE_VAD_FILTER:
        segments = intersect_segments(segments, vad_segments)

    # Step 1: Remove very short segments
    segments = filter_minimum_duration(segments, min_duration)
    
    # Step 2: Merge nearby segments from same speaker
    if use_adaptive:
        segments = adaptive_merge_segments(segments, base_gap=merge_gap)
    else:
        segments = simple_merge_segments(segments, gap_threshold=merge_gap)
    
    # Step 3: Apply boundary collar
    segments = apply_boundary_collar(segments, collar_seconds=collar)
    
    # Step 4: Final cleanup
    segments = filter_minimum_duration(segments, min_duration)
    
    return segments

print("✓ Intelligent post-processing functions defined")
print("  - Adaptive merging with context-aware thresholds")
print("  - Boundary collar for fuzzy transitions")
print("  - Multi-stage filtering")

## Step 6: Main Inference Pipeline

In [ ]:
def run_advanced_inference():
    """
    Advanced inference pipeline with all improvements.
    """
    # --- PYTORCH 2.6 SECURITY FIX ---
    _original_torch_load = torch.load
    def _trusted_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_torch_load(*args, **kwargs)
    torch.load = _trusted_load
    # --------------------------------

    print("="*70)
    print("🚀 ADVANCED BANGLA DIARIZATION PIPELINE")
    print("="*70)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🔧 Device: {device}")
    
    # 🆕 Load WhisperX's official Silero wrapper
    print("\n📥 Loading WhisperX Silero VAD model...")
    vad_model = Silero(
        device=torch.device(device),  
        vad_onset=VAD_ONSET,       
        vad_offset=VAD_OFFSET,
        chunk_size=30  # 🆕 The missing required parameter!
    )
    print("   ✓ WhisperX VAD loaded successfully")
    
    using_community = 1
    
    try:
        diarization_pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-community-1",  # 🆕 Updated model
            token=HF_TOKEN
        )
        FINETUNED_CKPT = "/kaggle/input/datasets/rubaiyatezaman/best-model-final-1/best_model (8).ckpt"
        finetuned_seg_model = Model.from_pretrained(FINETUNED_CKPT, map_location=device)
        print("   ✓ Diarization pipeline loaded (community - 1)")
        print(f"   ✓ Clustering threshold: {CLUSTERING_THRESHOLD}")
        print(f"   ✓ Min duration off: {MIN_DURATION_OFF}s")
        diarization_pipeline._segmentation.model = finetuned_seg_model
        diarization_pipeline.to(device)
        # Advanced instantiation with overlapped speech handling
        diarization_pipeline.instantiate({
            "clustering": {
                "threshold": CLUSTERING_THRESHOLD
            },
            "segmentation": {
                "min_duration_off": MIN_DURATION_OFF  # Detect brief pauses
            }
        })
        
    except Exception as e:
        using_community = 0
        diarization_pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1",
            use_auth_token=HF_TOKEN
        )
        FINETUNED_CKPT = "/kaggle/input/datasets/rubaiyatezaman/best-model-final-3-1/best_model (4).ckpt"
        finetuned_seg_model = Model.from_pretrained(FINETUNED_CKPT, map_location=device)
        print("\n📥 Loading pyannote/speaker-diarization-3.1...")
        print("   ✓ Diarization pipeline loaded (3.1)")
        print(f"   ✓ Clustering threshold: {CLUSTERING_THRESHOLD}")
        print(f"   ✓ Min duration off: {MIN_DURATION_OFF}s")
        diarization_pipeline._segmentation.model = finetuned_seg_model
        diarization_pipeline.to(device)
        diarization_pipeline.instantiate({
            "clustering": {
                "threshold": CLUSTERING_THRESHOLD
            },
            "segmentation": {
                "min_duration_off": MIN_DURATION_OFF  # Detect brief pauses
            }
        })
        # print(f"   ✗ Error loading pipeline: {e}")
        # print("   → Trying fallback to version 3.1...")
        # using_community = 0
        # diarization_pipeline = Pipeline.from_pretrained(
        #     "pyannote/speaker-diarization-3.1",
        #     use_auth_token=HF_TOKEN
        # )
        # print("   ✓ Loaded fallback version 3.1")

    
    
    # Get audio files
    audio_files = sorted(list(Path(TEST_AUDIO_DIR).glob("*.wav")))
    print(f"\n📂 Found {len(audio_files)} audio files to process")
    
    all_results = []
    preprocessed_files = []  # Track temp files for cleanup

    # Process each file
    print("\n" + "="*70)
    print("PROCESSING FILES")
    print("="*70)
    
    for i, audio_path in enumerate(audio_files):
        file_id = audio_path.stem
        print(f"\n[{i+1}/{len(audio_files)}] 🎵 {file_id}")
        
        try:
            # Step 1: Preprocess audio
            print("   → Preprocessing audio...")
            processed_path = preprocess_audio_advanced(str(audio_path), target_sr=TARGET_SAMPLE_RATE)
            if processed_path != str(audio_path):
                preprocessed_files.append(processed_path)
            
            # 🆕 Step 1.5: Run WhisperX VAD
            print("   → Running WhisperX VAD...")
            # WhisperX load_audio automatically converts to the required numpy array
            audio_array = whisperx.load_audio(processed_path)
            
            # FIX: vad_model directly returns the list of segments!
            whisperx_vad_segments = vad_model({
                "waveform": torch.from_numpy(audio_array).unsqueeze(0), 
                "sample_rate": TARGET_SAMPLE_RATE
            })

            print("   → Running Pyannote diarization...")
            # Step 2: Run diarization
            diarization = diarization_pipeline(processed_path)
            
            # Step 3: Extract raw segments
            raw_segments = []
            if using_community==0: 
                for turn, _, speaker in diarization.itertracks(yield_label=True):
                    raw_segments.append({
                        "start_time": round(turn.start, 2),
                        "end_time": round(turn.end, 2),
                        "speaker_id": speaker
                    })
            else: 
                for turn, speaker in diarization.exclusive_speaker_diarization:
                    raw_segments.append({
                        "start_time": round(turn.start, 2),
                        "end_time": round(turn.end, 2),
                        "speaker_id": speaker
                    })
                
            print(f"   → Raw Pyannote segments: {len(raw_segments)}")
            
            # Step 4: Apply advanced post-processing
            print("   → Applying post-processing & VAD intersection...")
            processed_segments = apply_postprocessing(
                raw_segments,
                min_duration=MIN_SPEECH_DURATION,
                vad_segments=whisperx_vad_segments,  # 🆕 Pass the WhisperX data here
                use_adaptive=USE_ADAPTIVE_MERGE,
                merge_gap=BASE_MERGE_GAP,
                collar=COLLAR_SECONDS
            )
            
            reduction = len(raw_segments) - len(processed_segments)
            print(f"   → Final segments: {len(processed_segments)} (-{reduction})")
            print(f"   ✓ Complete")
            # Save results
            all_results.append({
                "filename": file_id,
                "diarization": json.dumps(processed_segments)
            })
            
        except Exception as e:
            print(f"   ✗ Error: {e}")
            all_results.append({
                "filename": file_id,
                "diarization": "[]"
            })
    
    # Cleanup preprocessed files
    print("\n" + "="*70)
    print("CLEANUP")
    print("="*70)
    for temp_file in preprocessed_files:
        try:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        except:
            pass
    print(f"✓ Removed {len(preprocessed_files)} temporary files")
    
    # Save submission
    print("\n" + "="*70)
    print("SAVING RESULTS")
    print("="*70)
    
    df = pd.DataFrame(all_results)
    df.to_csv(OUTPUT_PATH, index=False)
    
    print(f"\n✅ SUCCESS! {OUTPUT_PATH} created with {len(df)} entries.")
    print("\n📋 First 3 entries:")
    print(df.head(3))
    print("\n" + "="*70)
    
    return df


print("✓ Advanced inference function defined")

## Step 7: Run Inference

In [ ]:
if __name__ == "__main__":
    results_df = run_advanced_inference()

## Step 8: Verify Output Format

In [ ]:
# Verify the submission format
df = pd.read_csv(OUTPUT_PATH)
print("📊 Submission Verification")
print("="*60)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSample entry (file: {df.iloc[0]['filename']}):")
print("="*60)

sample_diarization = json.loads(df.iloc[0]['diarization'])
for i, seg in enumerate(sample_diarization[:8], 1):  # Show first 8 segments
    duration = seg['end_time'] - seg['start_time']
    print(f"{i:2}. {seg['speaker_id']:15} [{seg['start_time']:6.2f}s - {seg['end_time']:6.2f}s] ({duration:.2f}s)")

if len(sample_diarization) > 8:
    print(f"    ... ({len(sample_diarization) - 8} more segments)")

print("\n✅ Output format verified!")

## Step 9: Statistics & Insights

In [ ]:
# Analyze diarization statistics
print("📈 Diarization Statistics")
print("="*60)

segment_counts = []
speaker_counts = []
avg_durations = []

for idx, row in df.iterrows():
    diar = json.loads(row['diarization'])
    if diar:
        segment_counts.append(len(diar))
        unique_speakers = len(set(seg['speaker_id'] for seg in diar))
        speaker_counts.append(unique_speakers)
        
        durations = [seg['end_time'] - seg['start_time'] for seg in diar]
        avg_durations.append(np.mean(durations) if durations else 0)

if segment_counts:
    print(f"Segments per file:")
    print(f"  Mean: {np.mean(segment_counts):.1f}")
    print(f"  Median: {np.median(segment_counts):.1f}")
    print(f"  Range: {min(segment_counts)} - {max(segment_counts)}")
    
    print(f"\nSpeakers per file:")
    print(f"  Mean: {np.mean(speaker_counts):.1f}")
    print(f"  Median: {np.median(speaker_counts):.1f}")
    print(f"  Range: {min(speaker_counts)} - {max(speaker_counts)}")
    
    print(f"\nAverage segment duration:")
    print(f"  Mean: {np.mean(avg_durations):.2f}s")
    print(f"  Median: {np.median(avg_durations):.2f}s")

print("\n" + "="*60)